In [18]:

import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import Input, LSTM, Dense, Embedding
from tensorflow.keras.models import Model

print("TensorFlow version:", tf.__version__)


TensorFlow version: 2.20.0


In [19]:
input_texts = [
    "good morning",
    "how are you",
    "see you later",
    "thank you",
    "i agree",
    "good night"
]

target_texts = [
    "buenos dias",
    "como estas",
    "hasta luego",
    "gracias",
    "si",
    "buenas noches"
]

print("Input texts :", input_texts)
print("Target texts:", target_texts)


Input texts : ['good morning', 'how are you', 'see you later', 'thank you', 'i agree', 'good night']
Target texts: ['buenos dias', 'como estas', 'hasta luego', 'gracias', 'si', 'buenas noches']


In [20]:

target_texts = ["\t" + text + "\n" for text in target_texts]

print("Target texts with start/end symbols:")
for t in target_texts:
    print(repr(t))


Target texts with start/end symbols:
'\tbuenos dias\n'
'\tcomo estas\n'
'\thasta luego\n'
'\tgracias\n'
'\tsi\n'
'\tbuenas noches\n'


In [21]:

input_chars = sorted(list(set("".join(input_texts))))
target_chars = sorted(list(set("".join(target_texts))))

input_char_to_index = {char: i + 1 for i, char in enumerate(input_chars)}
target_char_to_index = {char: i + 1 for i, char in enumerate(target_chars)}

input_index_to_char = {i: char for char, i in input_char_to_index.items()}
target_index_to_char = {i: char for char, i in target_char_to_index.items()}

num_encoder_tokens = len(input_chars) + 1
num_decoder_tokens = len(target_chars) + 1

print("Input characters :", input_chars)
print("Target characters:", target_chars)
print("Number of encoder tokens:", num_encoder_tokens)
print("Number of decoder tokens:", num_decoder_tokens)


Input characters : [' ', 'a', 'd', 'e', 'g', 'h', 'i', 'k', 'l', 'm', 'n', 'o', 'r', 's', 't', 'u', 'w', 'y']
Target characters: ['\t', '\n', ' ', 'a', 'b', 'c', 'd', 'e', 'g', 'h', 'i', 'l', 'm', 'n', 'o', 'r', 's', 't', 'u']
Number of encoder tokens: 19
Number of decoder tokens: 20


In [22]:

max_encoder_seq_length = max(len(text) for text in input_texts)
max_decoder_seq_length = max(len(text) for text in target_texts)

print("Max encoder sequence length:", max_encoder_seq_length)
print("Max decoder sequence length:", max_decoder_seq_length)


Max encoder sequence length: 13
Max decoder sequence length: 15


In [23]:

encoder_input_data = np.zeros((len(input_texts), max_encoder_seq_length), dtype="int32")
decoder_input_data = np.zeros((len(input_texts), max_decoder_seq_length), dtype="int32")
decoder_target_data = np.zeros((len(input_texts), max_decoder_seq_length, num_decoder_tokens), dtype="float32")

for i, (input_text, target_text) in enumerate(zip(input_texts, target_texts)):
    # Encoder input
    for t, char in enumerate(input_text):
        encoder_input_data[i, t] = input_char_to_index[char]

    # Decoder input and decoder target
    for t, char in enumerate(target_text):
        decoder_input_data[i, t] = target_char_to_index[char]
        if t > 0:
            decoder_target_data[i, t - 1, target_char_to_index[char]] = 1.0

print("Encoder input shape:", encoder_input_data.shape)
print("Decoder input shape:", decoder_input_data.shape)
print("Decoder target shape:", decoder_target_data.shape)


Encoder input shape: (6, 13)
Decoder input shape: (6, 15)
Decoder target shape: (6, 15, 20)


In [24]:

latent_dim = 64

encoder_inputs = Input(shape=(None,), name="encoder_inputs")
encoder_embedding = Embedding(input_dim=num_encoder_tokens, output_dim=16, mask_zero=True, name="encoder_embedding")(encoder_inputs)

encoder_lstm = LSTM(latent_dim, return_state=True, name="encoder_lstm")
encoder_outputs, state_h, state_c = encoder_lstm(encoder_embedding)

encoder_states = [state_h, state_c]


In [25]:

decoder_inputs = Input(shape=(None,), name="decoder_inputs")
decoder_embedding_layer = Embedding(input_dim=num_decoder_tokens, output_dim=16, mask_zero=True, name="decoder_embedding")
decoder_embedding = decoder_embedding_layer(decoder_inputs)

decoder_lstm = LSTM(latent_dim, return_sequences=True, return_state=True, name="decoder_lstm")
decoder_outputs, _, _ = decoder_lstm(decoder_embedding, initial_state=encoder_states)

decoder_dense = Dense(num_decoder_tokens, activation="softmax", name="decoder_dense")
decoder_outputs = decoder_dense(decoder_outputs)


In [26]:

model = Model([encoder_inputs, decoder_inputs], decoder_outputs)
model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])

model.summary()


Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ encoder_inputs      │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_inputs      │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_embedding   │ (None, None, 16)  │        304 │ encoder_inputs[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_3         │ (None, None)      │          0 │ encoder_inputs[0… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_embedding   │ (None, None, 16)  │        320 │ decoder_inputs[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_lstm (LSTM) │ [(None, 64),      │     20,736 │ encoder_embeddin… │
│                     │ (None, 64),       │            │ not_equal_3[0][0] │
│                     │ (None, 64)]       │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_lstm (LSTM) │ [(None, None,     │     20,736 │ decoder_embeddin… │
│                     │ 64), (None, 64),  │            │ encoder_lstm[0][… │
│                     │ (None, 64)]       │            │ encoder_lstm[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_dense       │ (None, None, 20)  │      1,300 │ decoder_lstm[0][… │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 43,396 (169.52 KB)

 Trainable params: 43,396 (169.52 KB)

 Non-trainable params: 0 (0.00 B)

In [27]:

history = model.fit(
    [encoder_input_data, decoder_input_data],
    decoder_target_data,
    batch_size=2,
    epochs=300,
    verbose=1
)


Epoch 1/300
3/3 ━━━━━━━━━━━━━━━━━━━━ 4s 22ms/step - accuracy: 0.0606 - loss: 2.7100
Epoch 2/300
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.1515 - loss: 2.7046
Epoch 3/300
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.1515 - loss: 2.6977
Epoch 4/300
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.1364 - loss: 2.6841
Epoch 5/300
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.1364 - loss: 2.6535
Epoch 6/300
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.1364 - loss: 2.6616
Epoch 7/300
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.1364 - loss: 2.6082
Epoch 8/300
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 0.1364 - loss: 2.5839
Epoch 9/300
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.1364 - loss: 2.5026
Epoch 10/300
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.1364 - loss: 2.4278
Epoch 11/300
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - accuracy: 0.1364 - loss: 2.4420
Epoch 12/300
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.1364 - lo

In [28]:

encoder_model = Model(encoder_inputs, encoder_states)


decoder_state_input_h = Input(shape=(latent_dim,), name="decoder_state_input_h")
decoder_state_input_c = Input(shape=(latent_dim,), name="decoder_state_input_c")
decoder_states_inputs = [decoder_state_input_h, decoder_state_input_c]

decoder_embedding_inf = decoder_embedding_layer(decoder_inputs)

decoder_outputs_inf, state_h_inf, state_c_inf = decoder_lstm(
    decoder_embedding_inf, initial_state=decoder_states_inputs
)

decoder_states_inf = [state_h_inf, state_c_inf]
decoder_outputs_inf = decoder_dense(decoder_outputs_inf)

decoder_model = Model(
    [decoder_inputs] + decoder_states_inputs,
    [decoder_outputs_inf] + decoder_states_inf
)


In [29]:

def decode_sequence(input_seq):
    states_value = encoder_model.predict(input_seq, verbose=0)

    target_seq = np.zeros((1, 1), dtype="int32")
    target_seq[0, 0] = target_char_to_index["\t"]

    decoded_sentence = ""

    while True:
        output_tokens, h, c = decoder_model.predict([target_seq] + states_value, verbose=0)

        sampled_token_index = np.argmax(output_tokens[0, -1, :])
        sampled_char = target_index_to_char.get(sampled_token_index, "")

        if sampled_char == "\n" or len(decoded_sentence) > max_decoder_seq_length:
            break

        decoded_sentence += sampled_char

        target_seq = np.zeros((1, 1), dtype="int32")
        target_seq[0, 0] = sampled_token_index

        states_value = [h, c]

    return decoded_sentence


In [30]:

for seq_index in range(len(test_sentences)):
    input_seq = encode_input_text(test_sentences[seq_index])
    decoded_sentence = decode_sequence(input_seq)

    print(f"Input: {test_sentences[seq_index]} --> Predicted Output: {decoded_sentence}")

Input: good morning --> Predicted Output: buenos dias
Input: how are you --> Predicted Output: como estas
Input: see you later --> Predicted Output: hasta luego
Input: thank you --> Predicted Output: gracias
Input: i agree --> Predicted Output: si
Input: good night --> Predicted Output: buenas noches


In [31]:

def encode_input_text(text):
    seq = np.zeros((1, max_encoder_seq_length), dtype="int32")

    for t, char in enumerate(text[:max_encoder_seq_length]):
        if char in input_char_to_index:
            seq[0, t] = input_char_to_index[char]

    return seq


test_sentences = [
    "good morning",
    "how are you",
    "see you later",
    "thank you",
    "i agree",
    "good night"
]

for sentence in test_sentences:
    encoded = encode_input_text(sentence)
    print(f"Input: {sentence} --> Output: {decode_sequence(encoded)}")


Input: good morning --> Output: buenos dias
Input: how are you --> Output: como estas
Input: see you later --> Output: hasta luego
Input: thank you --> Output: gracias
Input: i agree --> Output: si
Input: good night --> Output: buenas noches
